# Assignment 2 — Fine-tune a small open model (Typhoon) to match the A1 baseline

LoRA/QLoRA fine-tune of a Thai-first open model so a **self-hosted** model can match the Assignment-1
commercial baseline (`gemini-2.5-flash-lite`: amount-F1 100 / detail-F1 98.4) on the **same held-out
55-example eval set**.

**Runtime:** Colab, GPU = T4 (free). `Runtime → Change runtime type → T4 GPU`.

**What this notebook does**
1. Get the repo (train set + the A1 eval harness & held-out eval set — *single source of truth*).
2. Load Typhoon in 4-bit (Unsloth).
3. **Zero-shot eval** (before any training) — the required baseline row.
4. QLoRA fine-tune on `Assignment2/data/train.jsonl`.
5. **Fine-tuned eval** through the *same* A1 `run_model` harness.
6. 3-row comparison + per-bucket + failure analysis (wins / regressions / hard cases).
7. Save the LoRA adapter + reports to Google Drive.

> Methodology note: zero-shot and fine-tuned are scored by the **identical** A1 scoring code
> (`run_model` / `score_row` / `validate`), so the comparison is apples-to-apples.

In [ ]:
# 1a. Install (Colab). ~2-3 min.
# Let Unsloth pull a trl/peft/transformers set that is mutually compatible — do NOT pin an old trl
# (old trl passes tokenizer= to Trainer, which new transformers rejects). matplotlib is preinstalled.
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" 

In [ ]:
# 1b. Mount Google Drive (adapter + reports are saved here).
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/parnuan_a2'   # where the adapter + reports land
import os; os.makedirs(DRIVE_DIR, exist_ok=True)
print('saving outputs to', DRIVE_DIR)

In [ ]:
# 1c. Get the repo (train set + A1 eval harness + held-out eval set).
# Option A (preferred): clone from GitHub once the repo is published — set REPO_URL.
# Option B: if not published, upload the repo folder to Drive and set REPO_DIR to it.
import os, sys, pathlib

REPO_URL = "https://github.com/NaNiCatO/Parnuan-AI-Engineer-Intern.git"   # clones the repo; set "" to use REPO_DIR
REPO_DIR = "/content/drive/MyDrive/Parnuan AI Engineer Intern"            # used only if REPO_URL == ""

if REPO_URL:
    !rm -rf /content/repo && git clone --depth 1 {REPO_URL} /content/repo
    REPO = pathlib.Path("/content/repo")
else:
    REPO = pathlib.Path(REPO_DIR)

assert (REPO / "Assignment1" / "data" / "dataset.jsonl").exists(), f"eval set not found under {REPO}"
assert (REPO / "Assignment2" / "data" / "train.jsonl").exists(), f"train set not found under {REPO}"

# reuse A1's contract + scoring harness (single source of truth)
sys.path.insert(0, str(REPO / "Assignment1"))
from src.ner import validate, _parse_json, SYSTEM_PROMPT, ExtractResult
from src.eval import run_model, print_report
print("repo OK:", REPO)

In [ ]:
# 1d. Config
import json
# Free Colab has only ~12.7GB system RAM. Typhoon-8B has no pre-quantized build, so load_in_4bit must
# download the full ~16GB fp16 and convert it -> OOMs free Colab. We use a PRE-QUANTIZED 4-bit base
# (downloads ~5GB already-4bit, tiny RAM footprint). Qwen2.5-7B is small, multilingual (incl. Thai).
BASE_MODEL   = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"   # reliable on free T4
# Thai-first alternatives if you have >16GB RAM (Colab Pro / bigger GPU):
#   "scb10x/typhoon2-qwen2.5-7b-instruct"  (Qwen2.5 base, ChatML markers below already match)
#   "scb10x/llama3.1-typhoon2-8b-instruct" (Llama-3.1 base -> change markers in cell 4c)
MAX_SEQ_LEN  = 512
SEED         = 7

EVAL_PATH  = REPO / "Assignment1" / "data" / "dataset.jsonl"
TRAIN_PATH = REPO / "Assignment2" / "data" / "train.jsonl"

eval_rows  = [json.loads(l) for l in open(EVAL_PATH,  encoding="utf-8") if l.strip()]
train_rows = [json.loads(l) for l in open(TRAIN_PATH, encoding="utf-8") if l.strip()]
print(f"eval={len(eval_rows)} held-out  |  train={len(train_rows)}")

# leakage re-check inside the notebook (belt and suspenders)
def _norm(s): return " ".join(str(s).split()).strip().casefold()
_eval_set = {_norm(r["input"]) for r in eval_rows}
_leak = [r for r in train_rows if _norm(r["input"]) in _eval_set]
assert not _leak, f"LEAKAGE: {len(_leak)} train rows overlap the eval set!"
print("leakage check: 0 overlap")

In [ ]:
# 2. Load the base model in 4-bit (Unsloth)
import torch, time
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = True,
    dtype          = None,
)
print("loaded", BASE_MODEL)

In [ ]:
# 3a. Inference helper + extractor that plugs into A1's run_model.
#     Mirrors A1 graceful behaviour: empty/whitespace short-circuits to [].
def _generate(text: str) -> str:
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": text}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=256, do_sample=False,
                         use_cache=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

def hf_extractor(text, _model=None) -> ExtractResult:
    try:
        if not isinstance(text, str):
            text = "" if text is None else str(text)
        if not text.strip():
            return ExtractResult(short_circuited=True, route="short_circuit")
        if len(text) > 4000:
            text = text[:4000]
        t0 = time.perf_counter()
        raw = _generate(text)
        dt = (time.perf_counter() - t0) * 1000
        parsed = _parse_json(raw)
        if parsed is None:
            return ExtractResult(latency_ms=dt, error="unparseable_json")
        v = validate(parsed)
        return ExtractResult(transactions=v["transactions"], latency_ms=dt, error=None)
    except Exception as e:
        return ExtractResult(error=f"unexpected:{type(e).__name__}")

In [ ]:
# 3b. ZERO-SHOT eval (before any training) — the required baseline row.
FastLanguageModel.for_inference(model)
zs = run_model("typhoon-zeroshot", eval_rows, extractor=hf_extractor, label=f"{BASE_MODEL} (zero-shot)")
json.dump(zs, open(f"{DRIVE_DIR}/eval_zeroshot.json", "w"), ensure_ascii=False, indent=2)
print_report([zs])

In [ ]:
# 4a. Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED,
)

In [ ]:
# 4b. Tokenize with response-only masking (loss on the JSON completion only).
from datasets import Dataset
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build(ex):
    su   = [{"role":"system","content":SYSTEM_PROMPT}, {"role":"user","content":ex["input"]}]
    full = su + [{"role":"assistant","content":json.dumps({"transactions":ex["transactions"]}, ensure_ascii=False)}]
    pids = tokenizer.apply_chat_template(su,   tokenize=True, add_generation_prompt=True)
    fids = tokenizer.apply_chat_template(full, tokenize=True, add_generation_prompt=False)[:MAX_SEQ_LEN]
    labels = list(fids)
    for i in range(min(len(pids), len(fids))):   # mask the prompt; learn only the JSON
        labels[i] = -100
    return {"input_ids": fids, "attention_mask": [1]*len(fids), "labels": labels}

tok_ds = Dataset.from_list([build(e) for e in train_rows])
print(tok_ds, "| first example length:", len(tok_ds[0]["input_ids"]))

In [ ]:
# 4c. Train (QLoRA) with plain HF Trainer — version-robust, no trl dependency.
# ~10-20 min on a T4 for 1k examples x 2 epochs.
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported   # T4 (Turing) -> fp16; Ampere+ -> bf16

collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)
args = TrainingArguments(
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    warmup_steps=5, num_train_epochs=1, learning_rate=2e-4,
    logging_steps=5, optim="adamw_8bit", weight_decay=0.01,
    lr_scheduler_type="cosine", seed=SEED,
    fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
    output_dir="outputs", report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=tok_ds, data_collator=collator)
stats = trainer.train()
print(stats)

In [ ]:
# 4d. Loss curve -> Drive
import matplotlib.pyplot as plt
hist = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
xs, ys = zip(*hist)
plt.figure(figsize=(6,4)); plt.plot(xs, ys, marker="o", ms=3)
plt.xlabel("step"); plt.ylabel("training loss"); plt.title(f"LoRA loss — {BASE_MODEL}")
plt.grid(alpha=.3); plt.tight_layout(); plt.savefig(f"{DRIVE_DIR}/loss_curve.png", dpi=120)
plt.show()
print("final loss:", ys[-1])

In [ ]:
# 5. FINE-TUNED eval — same harness, same eval set.
FastLanguageModel.for_inference(model)
ft = run_model("typhoon-lora", eval_rows, extractor=hf_extractor, label=f"{BASE_MODEL} + LoRA")
json.dump(ft, open(f"{DRIVE_DIR}/eval_finetuned.json", "w"), ensure_ascii=False, indent=2)
print_report([ft])

In [ ]:
# 6a. Three-row comparison (A1 baseline + zero-shot + fine-tuned)
base = json.load(open(REPO / "Assignment1" / "reports" / "eval_google_gemini-2.5-flash-lite.json"))
def row(rep, name):
    o = rep["overall"]
    return f"| {name:<34} | {o['amount']['f1']*100:5.1f} | {o['detail']['f1']*100:5.1f} | {o['exact_match_rate']*100:5.1f} | {o['count_accuracy']*100:5.1f} |"
print("| Model                              | AmtF1 | DetF1 | Exact | Count |")
print("|------------------------------------|-------|-------|-------|-------|")
print(row(base, "A1 baseline: gemini-2.5-flash-lite"))
print(row(zs,   "Typhoon zero-shot"))
print(row(ft,   "Typhoon + LoRA"))

In [ ]:
# 6b. Failure analysis: wins / regressions / hard cases (baseline vs fine-tuned)
def by_input(rep): return {r["input"]: r for r in rep["rows"]}
b, f = by_input(base), by_input(ft)
def correct(r): return not r["errors"]
wins, regr, hard = [], [], []
for inp in set(b) & set(f):
    bc, fc = correct(b[inp]), correct(f[inp])
    if not bc and fc: wins.append(inp)
    elif bc and not fc: regr.append((inp, f[inp]["errors"]))
    elif not bc and not fc: hard.append(inp)
print(f"WINS (baseline wrong -> fine-tune right): {len(wins)}")
for i in wins[:8]: print("   +", i[:70])
print(f"\nREGRESSIONS (baseline right -> fine-tune wrong): {len(regr)}")
for i,e in regr[:8]: print("   -", i[:60], e)
print(f"\nHARD (both wrong): {len(hard)}")
for i in hard[:8]: print("   ?", i[:70])

In [ ]:
# 7. Save the LoRA adapter to Drive
ADAPTER_DIR = f"{DRIVE_DIR}/typhoon_lora_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("adapter saved to", ADAPTER_DIR)
print("\n>>> Paste back to finish A2: loss_curve.png, eval_zeroshot.json, eval_finetuned.json, the table above.")